In [1]:
import random
import torch
import os
import re

import pandas as pd
import polars as pl
import numpy as np

import sys
sys.path.append('../')
import compcor.corpus_metrics as corpus_metrics
from compcor.utils import Corpus
from compcor.text_tokenizer_embedder import STTokenizerEmbedder

from collections import defaultdict

/home/isabel/anaconda3/envs/localSyntheticData/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

In [2]:


# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)



In [3]:
# ------------------ Metric Setup (experiment_config.py) ------------------
comp_metrics = [
	corpus_metrics.chi_square_distance,
	corpus_metrics.zipf_distance,
	corpus_metrics.classifier_distance,
	corpus_metrics.IRPR_distance,
	corpus_metrics.fid_distance,
	corpus_metrics.pr_distance,
	corpus_metrics.dc_distance,
	corpus_metrics.mauve_distance,
	corpus_metrics.traditional_biber_distance,
	corpus_metrics.zero_wasserstein_distance
]

metrics_names = [((str(dist).split()[1]).split('_')[0]).upper() for dist in comp_metrics]

# Helper function for getting all metric data.
def get_data_for_compcor_metrics(corpus):
	tokens = STTokenizerEmbedder(embedding_model_name = "all-MiniLM-L12-v2").tokenize_sentences(corpus)
	embeddings = STTokenizerEmbedder(embedding_model_name = "all-MiniLM-L12-v2").embed_sentences(corpus)
	return tokens, embeddings

def get_distances_from_compare_corpora(setA, setB):    
    tokensA, embeddingsA = get_data_for_compcor_metrics(setA)
    tokensB, embeddingsB = get_data_for_compcor_metrics(setB)
    distances = {}
    for metric_name, metric in zip(metrics_names, comp_metrics):
        print(metric_name)
        if metric in (corpus_metrics.zipf_distance, corpus_metrics.chi_square_distance):
            tempA, tempB = tokensA, tokensB
        elif metric in (corpus_metrics.traditional_biber_distance, corpus_metrics.zero_wasserstein_distance):
            tempA, tempB = setA, setB
        else:
            tempA, tempB = embeddingsA, embeddingsB
        distances[metric_name] = metric(corpus1=tempA, corpus2=tempB)

    return distances


In [4]:
# subject the real data to the same processing as the generated data
def clean_note(text):
    text = re.sub(r'^[\s"]+|[\s"]+$', '', text)   # strip edge quotes/spaces
    text = re.sub(r'\*', '', text)                 # remove asterisks
    text = re.sub(r'\s+', ' ', text)              # normalize whitespace
    return text.strip()

real_datasets = {}
for folder in os.listdir('./getText/datasetsPrep/'):
    if os.path.isdir(f'./getText/datasetsPrep/{folder}'):
        for dataset in os.listdir(f'./getText/datasetsPrep/{folder}'):
            temp_df = pd.read_csv(f'./getText/datasetsPrep/{folder}/{dataset}', index_col=None)
            temp_df = temp_df.dropna(subset='text')
            temp_df['text'] = [clean_note(text) for text in temp_df['text'].tolist()]
            real_datasets[dataset.replace('.csv', '')] = temp_df

In [5]:
generated_datasets = {}
for dataset in os.listdir(f'./dataGeneration/processedData/'):
    temp_df = pd.read_csv(f'./dataGeneration/processedData/{dataset}', index_col=None)
    generated_datasets[dataset.replace('.csv', '')] = {
        'LDA': temp_df[temp_df['topic_model'] == 'LDA'],# ['report'].dropna().tolist(),
        'MATAVE': temp_df[temp_df['topic_model'] == 'MATAVE'],
        'combinedTopicModel': temp_df[temp_df['topic_model'].isin(['LDA', 'MATAVE'])]
    }

In [ ]:
assert generated_datasets.keys() == real_datasets.keys(), "Keys for both real and generated datasets must be the same."

dataset_metrics_averaged = {}
for dataset, topic_model_dict in generated_datasets.items():
    dataset_metrics_averaged[dataset] = {'LDA': {}, 'MATAVE': {}, 'combinedTopicModel': {}}
    temp_models = {'LDA': defaultdict(list), 'MATAVE': defaultdict(list), 'combinedTopicModel': defaultdict(list)}
    for i in range(3):
        real_sample = (real_datasets[dataset].sample(frac=1, random_state = i)['text'].tolist())[:100]
        for model in temp_models:
            assert len(topic_model_dict[model].dropna(subset=['report'])) >= 100
            model_sample = (topic_model_dict[model].sample(frac=1, random_state = i)['report'].dropna().tolist())[:100]
            temp_metrics = get_distances_from_compare_corpora(real_sample, model_sample)

            for metric, value in temp_metrics.items():
                temp_models[model][metric].append(value)
                                                  
    for model, metrics_dict in temp_models.items():
            for metric, values in metrics_dict.items():
                dataset_metrics_averaged[dataset][model][metric] = sum(values) / len(values)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

The `tokenize` method is deprecated, please use `preprocess` instead.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

CHI
ZIPF
CLASSIFIER
IRPR
FID
PR
Num real: 100 Num fake: 100
DC
Num real: 100 Num fake: 100
MAUVE


WARNING clustering 200 points to 10 centroids: please provide at least 390 training points


TRADITIONAL


[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_33_pied_piping']


ZERO


In [ ]:
for dataset, _ in generated_datasets.items():
    dataset_metrics_averaged[dataset]['realToReal'] = {}
    temp_models = {'realToReal': defaultdict(list)}
    for i in range(3):
        real_sample = real_datasets[dataset].sample(frac=1, random_state = i)['text'].tolist()
        assert len(real_sample) >= 200, "real samples must be over 200 samples long to prevent bias"
        real_start = real_sample[:100]
        real_end = real_sample[-100:]


        temp_metrics = get_distances_from_compare_corpora(real_start, real_end)

        for temp_metric, value in temp_metrics.items():
            temp_models['realToReal'][temp_metric].append(value)
                                                  
    for model, metrics_dict in temp_models.items():
            for temp_metric, values in metrics_dict.items():
                dataset_metrics_averaged[dataset][model][temp_metric] = sum(values) / len(values)

for dataset, _ in generated_datasets.items():
    dataset_metrics_averaged[dataset]['realToReal2'] = {}
    temp_models = {'realToReal2': defaultdict(list)}
    for i in range(3):
        real_sample = real_datasets[dataset].sample(frac=1, random_state = i + 200)['text'].tolist()
        assert len(real_sample) >= 200, "real samples must be over 200 samples long to prevent bias"
        real_start = real_sample[:100]
        real_end = real_sample[-100:]


        temp_metrics = get_distances_from_compare_corpora(real_start, real_end)

        for temp_metric, value in temp_metrics.items():
            temp_models['realToReal2'][temp_metric].append(value)
                                                  
    for model, metrics_dict in temp_models.items():
            for temp_metric, values in metrics_dict.items():
                dataset_metrics_averaged[dataset][model][temp_metric] = sum(values) / len(values)


for dataset, _ in generated_datasets.items():
    dataset_metrics_averaged[dataset]['realToReal3'] = {}
    temp_models = {'realToReal3': defaultdict(list)}
    for i in range(3):
        real_sample = real_datasets[dataset].sample(frac=1, random_state = i + 300)['text'].tolist()
        assert len(real_sample) >= 200, "real samples must be over 200 samples long to prevent bias"
        real_start = real_sample[:100]
        real_end = real_sample[-100:]


        temp_metrics = get_distances_from_compare_corpora(real_start, real_end)

        for temp_metric, value in temp_metrics.items():
            temp_models['realToReal3'][temp_metric].append(value)
                                                  
    for model, metrics_dict in temp_models.items():
            for temp_metric, values in metrics_dict.items():
                dataset_metrics_averaged[dataset][model][temp_metric] = sum(values) / len(values)

In [ ]:
rows = []
for dataset, model_dict in dataset_metrics_averaged.items():
    for model, metrics in model_dict.items():
        temp_dict = {
            'dataset': dataset,
            'model': model,

        }
        for metric, value in metrics.items():
            temp_dict[metric] = value
        rows.append(temp_dict)

df = pd.DataFrame(rows)
df.to_csv("./ldaMataveMetrics.csv", index=False)

In [ ]:
df.groupby('model').mean(numeric_only=True)